# 05. Exposing MCP Tools from an Azure Function App

This notebook annotates `function_app.py`, an **Azure Functions** app that exposes two tools — `get_order_status` and `list_customer_orders` — over the **Model Context Protocol (MCP)** using the `@app.mcp_tool_trigger` decorator, part of the Azure Functions Python worker's MCP extension. It belongs to Section 04, "Agent frameworks on top of Azure AI Foundry," in the `03-cloudxeus-mcp` sub-section — the section's example of *hosting* MCP tools on Azure infrastructure, as opposed to the earlier notebooks which *consumed* a model from a LangChain/LangGraph client.

**Difficulty: Advanced**

**Important:** this notebook is annotated, correct Python — but it cannot actually be *run* cell-by-cell inside Jupyter the way the earlier notebooks can. Azure Functions apps are event-driven server processes: `@app.mcp_tool_trigger` registers a function with the Functions host, but nothing invokes it until the host starts and an MCP client sends a request. There is no "print the result" step to run inline. See the Prerequisites section below for how to actually run and test this app.

## Prerequisites

**pip3 packages** (already listed in this folder's own `requirements.txt`, not the repo root one):
```bash
pip3 install azure-functions
```
(Optionally `azure-monitor-opentelemetry`, referenced but commented out in this folder's `requirements.txt`, if you want to wire this Function App to Application Insights the same way notebook 03 did for the LangChain agent.)

**Azure resources required:**
- An Azure Storage account (or the local Azurite emulator) for `AzureWebJobsStorage` — required by the Azure Functions runtime to manage triggers/bindings, even for HTTP/MCP-triggered functions.
- The **Azure Functions Core Tools** (`func` CLI) and the Python Functions runtime installed locally to run this app (`brew install azure-functions-core-tools@4` on macOS, or see Microsoft's install docs).
- To connect a real MCP client, you'd typically also deploy this Function App to Azure (`func azure functionapp publish <name>`) or run it locally and connect over `http://localhost:7071`.

**Configuration — not `.env`:** unlike every other notebook in this repo, an Azure Functions app is **not** configured via a root-level `.env` file loaded with `python-dotenv`. Its configuration lives in two function-app-specific files, both already present alongside `function_app.py` in `04. Section Code/03-cloudxeus-mcp/`:
- **`local.settings.json`** — local-only app settings (never deployed; excluded from source control in real projects via `.gitignore`). This app's copy sets:
  ```json
  {
    "IsEncrypted": false,
    "Values": {
      "AzureWebJobsStorage": "",
      "FUNCTIONS_WORKER_RUNTIME": "python"
    }
  }
  ```
  Fill in a real storage connection string (or `UseDevelopmentStorage=true` for Azurite) in `AzureWebJobsStorage` before running locally.
- **`host.json`** — runtime and extension-level configuration, including the `mcp` extension block that names the MCP server (`serverName: "TestServer"`, `serverVersion: "2.0.0"`), sets `encryptClientState`, and configures Application Insights sampling.

Both files are referenced here for context only, per this task's instructions — they are not modified or converted.

## What You'll Learn

- How `@app.mcp_tool_trigger` turns a plain Python function into an MCP tool served by Azure Functions
- The shape of `tool_properties` — a JSON schema describing a tool's expected arguments, analogous to the `@tool` docstring pattern seen in the LangChain notebooks, but declared explicitly instead of inferred from type hints
- Why an Azure Function app can't be "run" from within a notebook, and how to actually test it (`func start` + an MCP client)
- How this hosting model differs from calling tools client-side, as the LangChain/LangGraph notebooks in this section did

### Step 1 — Imports and the `FunctionApp` instance

`func.FunctionApp()` is the root object every trigger in this file attaches to via decorators. This mirrors FastAPI's `app = FastAPI()` pattern or Flask's `app = Flask(__name__)` — a single app object that collects all registered functions/routes/triggers for the runtime to discover.

💡 **Exam tip:** AI-102 doesn't test Azure Functions internals deeply, but it does expect you to recognize Azure Functions as a valid *hosting* choice for custom skills/tools that plug into Azure AI services (e.g., a custom skill in an Azure AI Search skillset — see this course's Section 07 `05_Lab - AI Enrichment and Skillsets/` — is commonly implemented as an Azure Function). MCP tool triggers are a newer variant of the same idea: Functions as the hosting layer for callable tool logic.

🔄 **Alternatives:** you can expose the same two tools as a standalone MCP server using the `mcp` Python SDK directly (`FastMCP` — see `10_mcp/02_mcp_server/` in this repo) running as a plain Python process, instead of hosting them inside Azure Functions. Azure Functions adds serverless scaling, built-in auth options, and Azure Monitor integration at the cost of the Functions programming model's constraints.

In [ ]:
import azure.functions as func
import json
import logging

app = func.FunctionApp()


### Step 2 — Tool 1: `get_order_status`

`order_status_properties` is a JSON-Schema-like list describing the tool's single argument (`order_id`, a required string) — this is what an MCP client sees when it lists available tools and their parameters, playing the same role the `@tool` docstring + type hints played in the LangChain notebooks, just declared explicitly rather than inferred.

`@app.mcp_tool_trigger(...)` registers `get_order_status` as an MCP tool named `"get_order_status"`. The function receives a single `context` argument (despite `arg_name="context"` suggesting otherwise, MCP tool triggers deliver the invocation payload as a JSON string, not a rich object) — the code parses it with `json.loads`, then defensively looks for the argument under a few possible key spellings (`order_id`, `orderId`, `order-id`) before falling back to `"UNKNOWN"`. The extensive `logging.info` calls are there specifically to debug the *shape* of that payload during development, since MCP payload structure can vary by client.

💡 **Exam tip:** this "defensive multi-key lookup" pattern is a sign of working against a loosely-typed integration boundary — a common real-world necessity when the schema on the wire isn't perfectly guaranteed. On AI-102, the analogous instinct applies to custom skill inputs/outputs in AI Search skillsets, which are similarly JSON-shaped and worth validating defensively.

🔄 **Alternatives:** if the MCP extension's Python bindings evolve to type the trigger's context automatically (some Functions trigger bindings support this, e.g., typed HTTP request objects), you could rely on that instead of manual `json.loads` + defensive key lookups.

In [ ]:
# --- Tool 1: Get Order Status ---
order_status_properties = json.dumps([
    {"name": "order_id", "type": "string",
     "description": "The order ID", "required": True}
])


@app.mcp_tool_trigger(
    arg_name="context",
    tool_name="get_order_status",
    description="Get the current status of a CloudXeus order.",
    tool_properties=order_status_properties,
)
def get_order_status(context) -> str:
    # Debug — log the full raw context so we can see its structure
    logging.info(f"RAW CONTEXT: {context}")

    content = json.loads(context)

    # Debug — log the parsed content
    logging.info(f"PARSED CONTENT: {json.dumps(content, indent=2)}")

    # Safe extraction with fallback
    arguments = content.get("arguments", content)
    order_id = arguments.get("order_id") or arguments.get("orderId") or arguments.get("order-id", "UNKNOWN")

    logging.info(f"Order lookup: {order_id}")
    return f"Order {order_id}: Shipped. Expected delivery: 2 days."


### Step 3 — Tool 2: `list_customer_orders`

Same pattern as Tool 1: a `tool_properties` schema describing the single `customer_id` argument, then an `@app.mcp_tool_trigger`-decorated function. This one returns a JSON string (a list of order records) rather than a plain sentence — MCP tool results are typically text, and returning `json.dumps(...)` is a common way to return structured data through a text-shaped protocol, letting the calling agent parse it back into structured data on its side.

💡 **Exam tip:** returning `json.dumps([...])` as a string (rather than a Python list) is deliberate — MCP tool return values are text content blocks. If you need a client to reliably parse the result as structured data, returning valid JSON text (as done here) is the correct approach, the same convention used by well-behaved LangChain `@tool` functions that need to return structured data.

🔄 **Alternatives:** for a tool with many possible query parameters (filters, pagination), you'd extend `tool_properties` with more entries and mark only the truly required ones `"required": True` — MCP clients use this to prompt the calling model correctly, and to validate arguments before invoking the tool.

In [ ]:
# --- Tool 2: List Orders by Customer ---
list_orders_properties = json.dumps([
    {"name": "customer_id", "type": "string",
     "description": "The customer ID", "required": True}
])


@app.mcp_tool_trigger(
    arg_name="context",
    tool_name="list_customer_orders",
    description="List all orders for a CloudXeus customer.",
    tool_properties=list_orders_properties,
)
def list_customer_orders(context) -> str:
    logging.info(f"RAW CONTEXT: {context}")

    content = json.loads(context)
    customer_id = content.get("arguments", content).get("customer_id", "UNKNOWN")

    logging.info(f"Listing orders for customer: {customer_id}")
    return json.dumps([
        {"order_id": "ORD-001", "status": "Shipped"},
        {"order_id": "ORD-002", "status": "Processing"},
    ])


### Step 4 — How to actually run and test this app

Since this is a server, not a script, testing it happens outside the notebook:

1. From `04. Section Code/03-cloudxeus-mcp/`, fill in a real `AzureWebJobsStorage` value in `local.settings.json` (or `"UseDevelopmentStorage=true"` if running Azurite locally).
2. Start the Functions host locally:
   ```bash
   func start
   ```
   This reads `host.json` (which names the MCP server `TestServer`, version `2.0.0`) and starts listening — by default at `http://localhost:7071`, with the MCP endpoint exposed as **streamable HTTP** per the Functions MCP extension's transport.
3. Connect an MCP client to that endpoint to list and call the two tools. The `mcp` Python SDK (already in the repo root `requirements.txt`) can act as that client — see `10_mcp/03_mcp_client/` in this repo for a working streamable-HTTP client pattern to adapt against this server's URL.
4. To deploy for real use (so an Azure AI Foundry agent or any remote MCP client can reach it), publish with `func azure functionapp publish <your-function-app-name>` after creating a Function App resource in Azure.

💡 **Exam tip:** AI-102 expects you to know how a Foundry Agent's tool layer can call out to *external* logic, and Azure Functions (via either plain HTTP-triggered functions wrapped as an OpenAPI/function tool, or this newer native MCP trigger) is one of the standard ways to host that logic on Azure without managing a VM or container yourself.

🔄 **Alternatives:** rather than the MCP-native trigger shown here, an equally valid (and more established) AI-102 pattern is an ordinary HTTP-triggered Azure Function described via an OpenAPI spec and registered as a Foundry Agent "OpenAPI tool" — functionally similar (Azure Functions hosting callable logic for an agent) but reached over plain HTTP + OpenAPI schema instead of the MCP protocol.

## Summary

This notebook annotated an Azure Functions app that hosts two MCP tools — `get_order_status` and `list_customer_orders` — using the `@app.mcp_tool_trigger` decorator from the Azure Functions Python MCP extension. Unlike the earlier notebooks in this section, there's no inline "run and see output" step: this is server code, configured via `host.json`/`local.settings.json` rather than `.env`, meant to be started with `func start` and exercised by a separate MCP client connecting over streamable HTTP. The core lesson is architectural — Azure Functions can *host* MCP tools for any MCP-aware agent to call, complementing the earlier notebooks where LangChain/LangGraph agents *consumed* tools defined client-side.

## Try It Yourself

1. **Easy:** Add a third tool, `cancel_order(order_id: str) -> str`, following the same `tool_properties` + `@app.mcp_tool_trigger` pattern as the two existing tools.
2. **Intermediate:** Run `func start` locally (with Azurite or a real storage account configured) and use an MCP client (e.g., adapt `10_mcp/03_mcp_client/`) to list the available tools and call `get_order_status` with a real `order_id`, observing the `logging.info` output in the `func` terminal.
3. **Advanced:** Wire this Function App to Application Insights (uncomment `azure-monitor-opentelemetry` in `requirements.txt` and configure `APPLICATIONINSIGHTS_CONNECTION_STRING` in `local.settings.json`), then compare the resulting traces to the LangChain-side tracing shown in notebook 03 of this section — same destination (Application Insights), two very different client-side and server-side origins.